# Semana 3 — Tarea del proyecto ancla · Aviso de horas fuera de especificación
### Cuaderno guiado (ALUMNO)

En el cuaderno satélite aprendiste a clasificar y a medir un clasificador: matriz de confusión, precisión, recall, F1 y curva ROC con su AUC. Ahora lo aplicas a la planta de flotación: **avisar, hora a hora, de si el concentrado va a salir fuera de especificación** (sílice por encima del 3 %) a partir de las variables de proceso, antes de que llegue el análisis del laboratorio.

**Sobre los datos.** Usamos la misma tabla horaria **semisintética** de la Semana 2 (`flotacion_semisintetica_horaria.csv`): las variables de proceso son las reales de la planta y la sílice del concentrado está simulada con un modelo del proceso que no se publica. La columna `fuera_spec` vale 1 cuando la sílice de esa hora supera el 3 %.

**Qué te damos hecho.** La sección 1 está resuelta: carga la tabla, fija las entradas y la salida y resume lo que conviene recordar de los datos. Solo tienes que ejecutarla y leerla.

**Qué entregas esta semana (tarea semanal)**

- Las **secciones 2 y 3**: una **regresión logística** evaluada con una **partición temporal** y comparada con la **línea base** (predecir siempre «dentro de especificación»). Completa las celdas `# TODO` y responde a las dos preguntas en las celdas «Tu respuesta».
- Se entrega como archivo `.ipynb` en el Campus Virtual, ejecutado de principio a fin sin errores.

**Qué no entregas esta semana.** La sección 4 es tu parte del **proyecto final** correspondiente a esta semana: una pregunta clave, una ampliación fundamental y dos ampliaciones más. No hay que hacerla ahora: se entrega con el proyecto final.

**Cómo trabajar**

- Ejecuta las celdas **en orden**, de arriba abajo.
- En las celdas con `# TODO` hay huecos marcados con `None` o con un comentario. Complétalos siguiendo la **pista** del comentario: casi siempre es una sola línea.
- Después de cada paso importante hay una celda **Comprueba**. Si da error, revisa ese paso antes de seguir: los pasos siguientes dependen de él.
- Un resultado modesto **no es un fracaso**: lo importante es que el método sea correcto y que interpretes el resultado con honestidad.

## 0. Preparación

Cargamos las librerías. Ejecuta la celda sin cambiar nada.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score)

pd.set_option('display.width', 120)

## 1. Los datos (resuelto)

Esta sección ya está hecha: ejecuta sus celdas sin cambiar nada y lee los resultados. Las secciones siguientes usan las variables que se definen aquí: `tabla`, `n_filas`, `objetivo` y `variables_proceso`.

### 1a. Cargar la tabla

La tabla tiene **una fila por hora** (marzo–septiembre de 2017). Cada fila trae la media horaria de las variables de proceso, la sílice de esa hora y la etiqueta `fuera_spec`.

| Columna | Qué contiene | Papel en la tarea |
|---|---|---|
| `date` | hora a la que corresponde la fila | metadato |
| `n_muestras` | número de registros de 20 s promediados en esa hora | metadato |
| `silice` | % de sílice en el concentrado (semisintética) | **prohibida**: de ella sale la etiqueta |
| `iron_feed`, `silica_feed` | % de hierro y de sílice del mineral que entra | entrada |
| `almidon_medio` | caudal de almidón | entrada |
| `amina_media` | caudal de amina | entrada |
| `caudal_pulpa` | caudal de pulpa | entrada |
| `ph_medio` | pH de la pulpa | entrada |
| `densidad` | densidad de la pulpa | entrada |
| `aire_medio` | aire inyectado, media de las 7 columnas de flotación | entrada |
| `nivel_medio` | nivel de espuma, media de las 7 columnas | entrada |
| `fuera_spec` | 1 si la sílice de esa hora supera el 3 %, 0 si no | **salida (lo que predecimos)** |

Los datos están en la carpeta `datasets`, al lado de la carpeta de esta semana. Si la celda siguiente da `FileNotFoundError`, revisa que has descargado el CSV en `SII/datasets/`.

In [ ]:
ruta_datos = '../datasets/flotacion_semisintetica_horaria.csv'
tabla = pd.read_csv(ruta_datos, parse_dates=['date'])

n_filas = tabla.shape[0]
n_columnas = tabla.shape[1]

print("Filas:", n_filas, "| columnas:", n_columnas)
tabla.head()

Comprobamos que has cargado el fichero correcto:

In [ ]:
assert n_filas == 4097, "Deberías tener 4097 filas: ¿has cargado flotacion_semisintetica_horaria.csv?"
assert n_columnas == 13, "Deberías tener 13 columnas"
print("Correcto")

### 1b. Entradas y salida (sin fugas)

El aviso tiene que llegar **antes** que el resultado del laboratorio. Por eso, como entradas solo valen las variables que la planta mide en tiempo real: las de papel «entrada» en la tabla de 1a.

Quedan **prohibidas** como entrada:

- `fuera_spec`: es lo que queremos predecir.
- `silice`: la etiqueta se calcula a partir de ella. Usarla sería darle al modelo la respuesta (**fuga de datos**).
- `date` y `n_muestras`: son metadatos, no describen el proceso.

> Regla práctica: si un modelo «acierta» demasiado, sospecha de una fuga.

In [ ]:
objetivo = 'fuera_spec'

variables_proceso = ['iron_feed', 'silica_feed', 'almidon_medio', 'amina_media',
                     'caudal_pulpa', 'ph_medio', 'densidad', 'aire_medio', 'nivel_medio']

print(len(variables_proceso), "entradas:", variables_proceso)

### 1c. Primer vistazo

La sílice a lo largo del tiempo, con el umbral de especificación (3 %), y cuántas horas quedan por encima:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(tabla['date'], tabla['silice'], lw=0.6)
ax.axhline(3.0, color='C3', ls='--', lw=1, label='umbral de especificación (3 %)')
ax.set_ylabel("% sílice")
ax.set_title("% de sílice a lo largo de los ~6 meses")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

n_fuera = tabla[objetivo].sum()
fraccion_fuera = tabla[objetivo].mean()
print(f"Horas fuera de especificación: {n_fuera} de {n_filas} ({fraccion_fuera:.1%})")

**Lo que conviene recordar de los datos**

- Una de cada cuatro horas está fuera de especificación: las dos clases están **desbalanceadas** (25 % frente a 75 %).
- La clase que nos interesa detectar, «fuera de especificación» (`fuera_spec = 1`), es la **minoritaria**. En las métricas la llamaremos la clase **positiva**.
- Como en la Semana 2, hay un hueco de 319 horas sin registros al principio. En la tarea no hay que hacer nada con él.

## 2. Dividir en train y test respetando el tiempo

Como en la Semana 2: entrenamos con las **primeras** horas (80 %) y evaluamos con las **últimas** (20 %), sin barajar. Dos horas seguidas de la planta son casi iguales; si se repartieran al azar, cada hora del test tendría a su vecina en el train y el modelo parecería mejor de lo que es.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `n_train` | número de horas del train (80 % de `n_filas`) |
| `train`, `test` | primeras `n_train` filas de `tabla` y el resto |
| `X_train_sin_escalar`, `X_test_sin_escalar` | las 9 entradas de proceso de cada parte, en sus unidades originales |
| `y_train`, `y_test` | etiqueta `fuera_spec` de cada parte (1 = fuera de especificación) |

In [ ]:
fraccion_train = 0.8
n_train = int(n_filas * fraccion_train)
train = tabla.iloc[:n_train]
test = tabla.iloc[n_train:]

X_train_sin_escalar = train[variables_proceso]
X_test_sin_escalar = test[variables_proceso]

# === TODO ===
# Guarda la salida (la columna objetivo) de cada parte.
# pista: y_train = train[objetivo]
y_train = None   # TODO
y_test = None    # TODO

fraccion_fuera_train = y_train.mean()
fraccion_fuera_test = y_test.mean()
print(f"train: {len(train)} horas, {fraccion_fuera_train:.1%} fuera de especificación")
print(f"test : {len(test)} horas, {fraccion_fuera_test:.1%} fuera de especificación")

**Comprueba:**

In [ ]:
assert len(y_train) == 3277 and len(y_test) == 820, "El train debe tener 3277 horas y el test 820"
assert y_test.sum() == 189, "En el test debería haber 189 horas fuera de especificación"
print("Correcto")

## 3. Regresión logística frente a la línea base

### 3a. Línea base: predecir siempre «dentro de especificación»

La regla más tonta posible es decir siempre que la hora va a salir bien (clase 0, la mayoritaria). Es nuestra **vara de medir**: un modelo solo vale la pena si la mejora.

Recuerda las métricas del satélite, con «fuera de especificación» como clase positiva:

| Métrica | Fórmula | Qué mide aquí |
|---|---|---|
| exactitud | (TP + TN) / total | fracción de horas bien clasificadas |
| precisión | TP / (TP + FP) | de las horas avisadas, cuántas eran de verdad malas |
| recall | TP / (TP + FN) | de las horas malas, cuántas avisamos |
| F1 | 2 · precisión · recall / (precisión + recall) | resumen de precisión y recall |

In [ ]:
n_test = len(y_test)
prediccion_base = np.zeros(n_test, dtype=int)   # 0 = dentro de especificación, para todas las horas

# === TODO ===
# Calcula la exactitud y el recall de la línea base en el test.
# pista: accuracy_score(y_test, prediccion_base)   y   recall_score(y_test, prediccion_base)
exactitud_base = None   # TODO
recall_base = None      # TODO

print(f"LÍNEA BASE   exactitud = {exactitud_base:.3f}   recall = {recall_base:.3f}")

**Comprueba:** deberías obtener una exactitud de la línea base de aproximadamente 0,77 y un recall de 0.

In [ ]:
assert abs(exactitud_base - 0.770) < 0.005, "Revisa la exactitud de la línea base: compara y_test con prediccion_base"
assert recall_base == 0, "La línea base no avisa nunca: su recall debe ser 0"
print("Correcto")

### 3b. Regresión logística

Estandarizamos las entradas con el train, como en la Semana 2, y entrenamos una `LogisticRegression`. La logística da, para cada hora, la **probabilidad** de que esté fuera de especificación; con el umbral 0,5 la convertimos en una predicción (1 si la probabilidad llega a 0,5).

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `escalador` | `StandardScaler` ajustado **solo** con `X_train_sin_escalar` |
| `X_train`, `X_test` | entradas estandarizadas |
| `modelo` | la regresión logística entrenada |
| `probabilidad_test` | probabilidad de «fuera de especificación» de cada hora del test |
| `prediccion_test` | 1 si `probabilidad_test` llega a 0,5; 0 si no |

In [ ]:
escalador = StandardScaler()
escalador.fit(X_train_sin_escalar)   # solo con el train
X_train = escalador.transform(X_train_sin_escalar)
X_test = escalador.transform(X_test_sin_escalar)

modelo = LogisticRegression(max_iter=2000)

# === TODO ===
# 1) Entrena el modelo con X_train e y_train (una línea).   pista: modelo.fit(X_train, y_train)
# 2) Calcula las probabilidades del test.                   pista: probabilidades = modelo.predict_proba(X_test)


probabilidades = None   # TODO

probabilidad_test = probabilidades[:, 1]                   # columna 1 = probabilidad de fuera de especificación
prediccion_test = (probabilidad_test >= 0.5).astype(int)   # umbral 0,5
print("Primeras 5 probabilidades del test:", probabilidad_test[:5].round(2))

### 3c. Matriz de confusión y métricas

La matriz de confusión del test (filas = clase real, columnas = clase predicha). Ejecuta sin cambiar nada:

In [ ]:
matriz_confusion = confusion_matrix(y_test, prediccion_test)
grafico_matriz = ConfusionMatrixDisplay(matriz_confusion, display_labels=["dentro", "fuera"])
grafico_matriz.plot(cmap="Blues", colorbar=False)
plt.title("Matriz de confusión en el test (umbral 0,5)")
plt.xlabel("clase predicha")
plt.ylabel("clase real")
plt.tight_layout()
plt.show()

In [ ]:
# === TODO ===
# Calcula las métricas del modelo en el TEST, con «fuera de especificación» (1) como clase positiva.
# pista: accuracy_score(y_test, prediccion_test); para el AUC usa las probabilidades: roc_auc_score(y_test, probabilidad_test)
exactitud = None   # TODO
precision = None   # TODO
recall = None      # TODO
f1 = None          # TODO
auc = None         # TODO

print(f"{'':12s} {'exactitud':>9s} {'precisión':>9s} {'recall':>7s} {'F1':>6s} {'AUC':>6s}")
print(f"{'línea base':12s} {exactitud_base:9.3f} {'—':>9s} {recall_base:7.3f} {0:6.3f} {0.5:6.3f}")
print(f"{'logística':12s} {exactitud:9.3f} {precision:9.3f} {recall:7.3f} {f1:6.3f} {auc:6.3f}")

**Comprueba:** deberías obtener, en el test, una exactitud de aproximadamente 0,80, un recall de aproximadamente 0,19 y un AUC de aproximadamente 0,74.

In [ ]:
assert abs(exactitud - 0.800) < 0.005, "Revisa las métricas: la exactitud del test debería ser aproximadamente 0,80"
assert abs(recall - 0.190) < 0.005, "Revisa las métricas: el recall debería ser aproximadamente 0,19"
assert abs(auc - 0.737) < 0.005, "Revisa el AUC: usa probabilidad_test, no prediccion_test"
print("Correcto")

**Pregunta 3.1.** ¿Bate tu modelo a la línea base? Compara las dos en exactitud y en recall. Con la matriz de confusión, ¿cuántas de las horas que de verdad salieron fuera de especificación avisa el modelo y cuántas se le escapan?

**Tu respuesta:** *escribe aquí*

**Pregunta 3.2.** Explica con palabras, pensando en el operador de la planta, qué significan la precisión y el recall que has obtenido. ¿Le sirve este aviso tal como está? ¿Qué podrías cambiar para que avisara de más horas malas sin entrenar otro modelo, y qué perderías a cambio?

**Tu respuesta:** *escribe aquí*

> **Antes de entregar:** reinicia el kernel y ejecuta todo (en VS Code, «Restart» y luego «Run All»; en Colab, «Entorno de ejecución → Reiniciar y ejecutar todo»). Comprueba que no hay errores y que todas las celdas «Comprueba» dicen «Correcto». Guarda y sube el `.ipynb` al Campus Virtual.

## 4. Proyecto final

Esta sección **no se entrega esta semana**. Es tu parte del **proyecto final** correspondiente a la Semana 3: puedes hacerla durante el curso o en el mes de trabajo autónomo, y se entrega con el proyecto final.

> **Recordatorio: las dos preguntas del proyecto**
>
> - **Pregunta general del curso:** *¿Puedo dar al operador una estimación fiable de la sílice y una recomendación de consignas?* Cada semana responde a una parte; en el resumen final reúnes lo que has aprendido en todas.
> - **Pregunta clave de esta semana:** *¿Puede el sistema avisar al operador de las horas fuera de especificación antes de que llegue el laboratorio, y a cambio de cuántas falsas alarmas?* Es la parte de la pregunta general que toca en la Semana 3: convertir la estimación en un aviso.

Aquí no hay un guion de pasos. Te decimos **qué buscamos** y te damos **pistas** de por dónde ir; cómo lo resuelvas es cosa tuya. Lo importante no es ganar, sino que cada afirmación esté respaldada por un experimento y que expliques por qué una opción funciona mejor o peor.

### Qué buscamos

Un **sistema de aviso** para el operador que:

- use como entradas **solo variables de proceso**, y ajuste el escalador y cualquier otra transformación **solo con los datos con los que entrena**;
- se evalúe con la **partición temporal** de la tarea semanal y **frente a la línea base**;
- tome **todas sus decisiones** (entradas, hiperparámetros, umbral de aviso…) **sin mirar el test**, con la validación temporal de abajo, y use el test **una sola vez** al final;
- decida cuándo avisar con un **criterio de coste explícito**. Como referencia, dejar escapar una hora fuera de especificación cuesta como **5 falsas alarmas**: una hora mala produce concentrado que hay que reprocesar, mezclar o vender con penalización, mientras que una falsa alarma solo le cuesta al operador revisar la planta unos minutos;
- termine con una **respuesta para el operador**: de qué fracción de las horas malas le avisa y cuántas falsas alarmas recibe al día.

El trabajo tiene tres niveles, y cada uno se apoya en el anterior: la ampliación fundamental (4.1) para alcanzar el aprobado, la comparación de técnicas (4.2) para alcanzar el notable y la propuesta de una técnica nueva (4.3) para alcanzar el sobresaliente.

### Validación temporal (resuelto)

Para elegir sin mirar el test, reservamos las **últimas horas del train** como validación, sin barajar, como en la Semana 2. Entrena cada opción con `train_ajuste`, compárala en `validacion`, y entrena la elegida con todo el `train` antes de evaluarla en el `test`.

| Variable | Qué contiene |
|---|---|
| `train_ajuste` | primeras horas del train (80 %): para entrenar mientras comparas opciones |
| `validacion` | últimas horas del train (20 %): para elegir sin mirar el test |

Ejecuta la celda sin cambiar nada.

In [ ]:
fraccion_validacion = 0.2
n_validacion = int(len(train) * fraccion_validacion)
n_ajuste = len(train) - n_validacion

train_ajuste = train.iloc[:n_ajuste]
validacion = train.iloc[n_ajuste:]

inicio_validacion = validacion['date'].min()
fin_validacion = validacion['date'].max()
n_fuera_validacion = validacion[objetivo].sum()
print(f"train_ajuste: {len(train_ajuste)} horas")
print(f"validacion  : {len(validacion)} horas  ({inicio_validacion:%Y-%m-%d} -> {fin_validacion:%Y-%m-%d}), {n_fuera_validacion} fuera de especificación")
print(f"test        : {len(test)} horas")

### 4.1 Ampliación fundamental (para aprobado)

**Objetivo.** Mejorar el aviso de la tarea semanal **sin salir de la regresión logística**, interpretar con precisión lo que mide y convertirlo en una decisión: cuándo avisar.

**Qué buscamos.** Una logística mejor que la de la tarea semanal, elegida en la validación; sus métricas en el test explicadas **en horas de planta** (qué cuenta cada una, qué significa su valor y qué no te dice); un umbral de aviso elegido con el criterio de coste; y la respuesta para el operador.

**Pistas**

- La logística de la tarea semanal es lineal en las entradas. En la Semana 2 viste qué hacer cuando un modelo lineal se queda corto: ¿y si le das cuadrados y productos de las entradas (`PolynomialFeatures`)?
- El umbral 0,5 no es obligatorio. `predict_proba` te da la probabilidad de cada hora; con ella puedes recorrer umbrales y contar, para cada uno, horas malas que se escapan y falsas alarmas.
- `roc_curve`, `precision_recall_curve` y `ConfusionMatrixDisplay`, de `sklearn.metrics`, te ayudan a ver el compromiso entre avisar más y equivocarse más.
- `class_weight='balanced'` es otra forma de que el modelo preste más atención a la clase minoritaria. ¿Es lo mismo que mover el umbral?

In [ ]:
# Tu código para 4.1


**Tu respuesta (4.1):** *escribe aquí*

### 4.2 Comparación de técnicas (para notable)

**Objetivo.** Saber si **otras técnicas de clasificación** avisan mejor que tu logística de 4.1, y por qué.

**Qué buscamos.** Una comparación en igualdad de condiciones (mismas entradas disponibles, misma validación para elegir, mismo criterio de coste, mismo test) entre tu mejor logística y las técnicas de la asignatura que se puedan aplicar, con una explicación de por qué cada una funciona mejor o peor.

**Pistas**

- Árboles de decisión (`DecisionTreeClassifier`) y métodos agregados (`RandomForestClassifier`, `HistGradientBoostingClassifier`). Sus hiperparámetros, como la profundidad o el tamaño mínimo de las hojas, también se eligen en la validación.
- Compara el resultado en el train y en la validación: ¿alguna técnica se sobreentrena?
- El AUC compara sin elegir umbral; el coste compara con el umbral elegido. ¿Ordenan igual las técnicas?
- En la Semana 2 construiste un estimador de la sílice. Avisar cuando la sílice estimada pase del 3 % también es un clasificador.
- La tabla enriquecida (`flotacion_semisintetica_horaria_enriquecida.csv`) trae el aire y el nivel de cada columna de flotación por separado. ¿A qué técnicas les sirve?

In [ ]:
# Tu código para 4.2


**Tu respuesta (4.2):** *escribe aquí*

### 4.3 Propuesta de una técnica nueva (para sobresaliente)

**Objetivo.** Proponer y probar **una idea que no hayamos visto en clase** y que tenga sentido para la pregunta de la semana.

**Qué buscamos.** Que justifiques por qué la eliges, que la compares con tus mejores modelos anteriores con la misma evaluación temporal y el mismo criterio de coste, y que expliques por qué funciona mejor o peor. Si no mejora, también vale: explica por qué.

**Pistas** (ideas, no una lista cerrada)

- Otros clasificadores: máquinas de vectores soporte, k vecinos, redes neuronales…
- Probabilidades en las que el operador pueda confiar: si el sistema dice «70 % de probabilidad de fuera de especificación», ¿se cumple el 70 % de las veces?
- Avisos con más de dos niveles (por ejemplo, verde, ámbar y rojo) en lugar de un sí o no.
- Usar lo que hicieron las variables de proceso en las horas anteriores, sin cruzar el hueco de registros.

In [ ]:
# Tu código para 4.3


**Tu respuesta (4.3):** *escribe aquí*